# Stock Market Sentiment Analysis Using Tweets

**Text Mining group project — NOVA IMS, 2024/2025**

**Contributors:** Ana Caleiro, André Lourenço, Duarte Marques, Érica Parracho, and Jude Gbenimako.

This notebook is a cleaned archival copy of the original academic work. Saved execution outputs have been removed; original training and test CSVs are not distributed in this portfolio repository. The code has not been re-executed or updated for current library compatibility.


## Selected previously recorded experimental results

The original notebook recorded **accuracy 0.8072** and **macro F1 0.7199** for the best Logistic Regression + Bag-of-Words experiment (tweet-tailored cleaning). Other vectorization and classification approaches were explored. The original saved execution outputs have been cleared for public presentation; this is an archival result, **not re-executed** for this copy.


# **Table of Contents**

>[0. Context and Metadata](#scrollTo=jL7ySmXoqWL8)

>[1. Importing Libraries and Loading Data](#scrollTo=QIPp1Gn6q_cV)

>>[1.1. Importing Libraries](#scrollTo=YukjMiNRrAO5)

>>[1.2. Functions](#scrollTo=UT6EXFsDrIRK)

>>[1.3. Loading and Reading the Data](#scrollTo=Dg5kPVdUrNH1)

>[2. Preprocessing](#scrollTo=f8ad6745)

>>[2.1. Cleaning Approaches](#scrollTo=lPpSzvBTYOgp)

>>>[2.1.1. Class inspired Cleaning](#scrollTo=BvI9b7YeYXo7)

>>>[2.1.2. Tailored to Tweets and Social](#scrollTo=XJlWmyymYdYX)

>>>[2.1.3. Unified Methods](#scrollTo=vVz5QEGmYroE)

>>[2.2. More Data Exploration](#scrollTo=eTcoRfbsj-Wx)

>>[2.3. More Preprocessing](#scrollTo=R3dOVDLFlZmr)

>>>[2.3.1 Word tokenize](#scrollTo=CZdAnmli4v4P)

>>>[2.3.2. BertTokenizerFas](#scrollTo=ov4p3zj240V1)

>[3. Feature Engineering](#scrollTo=kyf_rfHBY3Y2)

>>[3.1 Train and Evaluation Split](#scrollTo=hhJhko32axKp)

>>[3.2 IF-IDF Vectorization](#scrollTo=zyIXqkIxtWXy)

>>[3.3 Bag of Words Vectorization](#scrollTo=lMkpgsAkZJf8)

>>[3.4 Word 2 Vectorization](#scrollTo=V_TfJ21daIZW)

>>[3.5. Sentence Transformer](#scrollTo=NqbnPEgpCjHB)

>[4. Modeling](#scrollTo=q6fEdxt-tlgt)

>>[4.1 Modeling with IF-IDF](#scrollTo=Z2FSIZn7cir_)

>>>[4.1.1 Essemble Voting](#scrollTo=6cGzHYRBFYE2)

>>>[4.1.2 Grid Search SVM and Smoote](#scrollTo=8p-p57r4oR2V)

>>[4.2 Modeling with Word 2 Vec](#scrollTo=hflnUUAwH2E4)

>>[4.3 Modeling with Bag of Words](#scrollTo=iDAEbQrlUAD5)

>>[4.4. Modeling with SBert](#scrollTo=Qpr0Lb2vUnd2)

>>[4.5. Modeling with Transformers Classifiers](#scrollTo=4n_yNhtppHLW)

>[5. Evaluation](#scrollTo=h9MYVk4bXPmG)



# **0. Context and Metadata**

We are performing a sentiment analysis of the stock market, predicting its behavior based on tweets. To accomplish this, we will use a classification model and NLP techniques to speculate the sentiment of each tweet (Bearish (0), Bullish (1), or Neutral (2)).

**Purpose of the notebook:**

In this notebook, we have all the techniques we experimented to get to the final predictions.



# **1. Importing Libraries and Loading Data**


## **1.1. Importing Libraries**


In [ ]:
#!pip install gensim
#!pip install xgboost
#!pip install contractions
#pip install pyspellchecker
#!pip install sentence-transformers

import numpy as np
import pandas as pd
import string
import re
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm import tqdm, trange
import random

# Text Preprocessing
import nltk
import contractions
from spellchecker import SpellChecker
from wordcloud import WordCloud

from nltk.tokenize import word_tokenize
from nltk import tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.probability import FreqDist

# Feature Extraction
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Modeling
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

# Imbalanced Data Handling
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline

# Transformers
from transformers import BertTokenizerFast
from transformers import TFBertModel
from transformers import RobertaTokenizerFast
from transformers import TFRobertaModel

# Ensuring Consistency
random.seed(11)
np.random.seed(11)

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')
nltk.download('wordnet')

## **1.2. Functions**

Preprocessing

In [ ]:
def number_words(tweet_text):
    text_list = tweet_text.split()
    return(len(text_list))

def all_upper(tweet_text):
    uppercase_words = [word for word in tweet_text.split() if word.isupper()]
    return len(uppercase_words)

def count_sentence(tweet_text):
    return len(nltk.sent_tokenize(tweet_text))

def count_unique_words(tweet_text):
    return len(set(tweet_text.split()))

def count_htags(tweet_text):
    count = re.findall(r'\#[a-zA-Z0-9]+', tweet_text)
    return len(count)

def count_mentions(tweet_text):
	count = re.findall(r'\@[A-Za-z0-9]+', tweet_text)
	return len(count)

def count_index(tweet_text):
    count = re.findall(r'\$[a-zA-Z]+', tweet_text)
    return len(count)

def count_value(tweet_text):
	x = re.findall(r'\$[0-9]+', tweet_text)
	return len(x)

def count_stopwords(tweet_text):
	stop_words = set(stopwords.words('english'))
	word_tokens = word_tokenize(tweet_text)
	stopwords_x = [w for w in word_tokens if w in stop_words]
	return len(stopwords_x)



In [ ]:
# More specified cleaning techniques

#Clean emojis from text
def strip_emoji(text):
    return re.sub(emoji.get_emoji_regexp(), r"", text) #remove emoji

#Remove punctuations, links, mentions and \r\n new line characters
def strip_all_entities(text):
    text = text.replace('\r', '').replace('\n', ' ').replace('\n', ' ').lower() #remove \n and \r and lowercase
    text = re.sub(r"(?:\@|https?\://)\S+", "", text) #remove links and mentions
    text = re.sub(r'[^\x00-\x7f]',r'', text) #remove non utf8/ascii characters such as '\x9a\x91\x97\x9a\x97'
    banned_list= string.punctuation + 'Ã'+'±'+'ã'+'¼'+'â'+'»'+'§'
    table = str.maketrans('', '', banned_list)
    text = text.translate(table)
    return text

#clean hashtags at the end of the sentence, and keep those in the middle of the sentence by removing just the # symbol
def clean_hashtags(tweet):
    new_tweet = " ".join(word.strip() for word in re.split('#(?!(?:hashtag)\b)[\w-]+(?=(?:\s+#[\w-]+)*\s*$)', tweet)) #remove last hashtags
    new_tweet2 = " ".join(word.strip() for word in re.split('#|_', new_tweet)) #remove hashtags symbol from words in the middle of the sentence
    return new_tweet2

#Filter special characters such as & and $ present in some words
def filter_chars(a):
    sent = []
    for word in a.split(' '):
        if ('$' in word) | ('&' in word):
            sent.append('')
        else:
            sent.append(word)
    return ' '.join(sent)

def remove_mult_spaces(text): # remove multiple spaces
    return re.sub("\s\s+" , " ", text)

In [ ]:
# Class inpired Cleaning

def clean(text_list, lemmatize, stemmer):

    updates = []

    for j in tqdm(text_list):

        text = j

        #LOWERCASE TEXT
        text = text.lower()

        #REMOVE NUMERICAL DATA and PUNCTUATION
        text = re.sub("[^a-zA-Z]"," ", text )
        # [^&] - Returns a match for any character EXCEPT &.
        # a-zA-Z - Returns a match for any character alphabetically between a and z, lower case or upper case.
        #text = re.sub("br", "", text)  # remove "br"
        text = re.sub(r'http\S+', '', text)  # remove URLs
        text = re.sub(f"[{re.escape(string.punctuation)}]", '', text)  # remove punctuation

        text = re.sub(r'\bco\b', '', text) #remove co in isolation
        text = re.sub(r'\s+', ' ', text).strip()


        #REMOVE STOPWORDS
        text = " ".join([word for word in text.split() if word not in stop])

        #Lemmatize
        if lemmatize:
            text = " ".join(lemma.lemmatize(word) for word in text.split())

        #Stemming
        if stemmer:
            text = " ".join(stemmer.stem(word) for word in text.split())

        updates.append(text)

    return updates

In [ ]:
# Merged Cleaning approaches

def clean_financial_tweet_with_tickers(text, lemmatize, stem):
    # 0. Extract cashtags (e.g., $TSLA)
    tickers = re.findall(r'\$\w+', text.upper())  # captures tickers in uppercase
    tickers = [t[1:] for t in tickers]  # remove the '$'

    # 1. Remove emojis
    text = re.sub(emoji.get_emoji_regexp(), "", text)

    # 2. Remove mentions, links, non-ascii chars, lowercase
    text = text.replace('\r', '').replace('\n', ' ').lower()
    text = re.sub(r"(?:\@|https?\://)\S+", "", text)
    text = re.sub(r'[^\x00-\x7f]',r'', text)

    # 3. Remove punctuation and weird chars
    banned_list = string.punctuation + 'Ã'+'±'+'ã'+'¼'+'â'+'»'+'§'
    text = text.translate(str.maketrans('', '', banned_list))

    # 4. Clean hashtags
    text = " ".join(word.strip() for word in re.split('#(?!(?:hashtag)\b)[\w-]+(?=(?:\s+#[\w-]+)*\s*$)', text))
    text = " ".join(word.strip() for word in re.split('#|_', text))

    # 5. Keep $ words intact (already extracted)
    # Optionally replace them with a placeholder like "TICKER" in text
    text = re.sub(r'\$\w+', '', text)

    # 6. Remove numbers and clean up
    text = re.sub("[^a-zA-Z]", " ", text)
    text = re.sub(r'\bco\b', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # 7. Stopwords
    text = " ".join(word for word in text.split() if word not in stop)

    # 8. Lemmatize
    if lemmatize:
        text = " ".join(lemma.lemmatize(word) for word in text.split())

    # 9. Stemming
    if stem:
        text = " ".join(stemmer.stem(word) for word in text.split())

    return text, tickers


Modelling


In [ ]:
def run_tfidf_classification(datasets, max_features=10000):
    """
    Run classification on different perspectives using TF-IDF.
    """
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.svm import LinearSVC
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from xgboost import XGBClassifier
    import seaborn as sns
    import matplotlib.pyplot as plt

    models = {
        'NaiveBayes': MultinomialNB(),
        'SVM': LinearSVC(),
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
        'XGBoost': XGBClassifier(eval_metric='mlogloss', use_label_encoder=False, random_state=42),
        'LogisticRegression': LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=500, random_state=42),
    }

    all_results = {}
    best_models_summary = []

    for name, df in datasets.items():
        print(f"\n=== Dataset: {name} ===")
        X = df["text_no_contractions"]
        y = df["label"]

        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

        # TF-IDF vectorization
        vectorizer = TfidfVectorizer(max_features=max_features)
        X_train_vec = vectorizer.fit_transform(X_train)
        X_val_vec = vectorizer.transform(X_val)

        perspective_results = {}
        best_model = None
        best_f1 = -1

        for model_name, model in models.items():
            print(f"\nModel: {model_name}")

            model.fit(X_train_vec, y_train)
            y_pred = model.predict(X_val_vec)

            acc = accuracy_score(y_val, y_pred)
            f1 = f1_score(y_val, y_pred, average='macro')

            # Save results for later summary
            perspective_results[model_name] = {
                "accuracy": acc,
                "f1_macro": f1,
                "report": classification_report(y_val, y_pred, output_dict=True)
            }

            # Save best-performing model
            if f1 > best_f1:
                best_model = model_name
                best_f1 = f1
                best_acc = acc

            # Save to globals
            globals()[f"{model_name}_TFIDF_accuracy"] = max(globals().get(f"{model_name}_TFIDF_accuracy", 0), acc)
            globals()[f"{model_name}_TFIDF_f1"] = max(globals().get(f"{model_name}_TFIDF_f1", 0), f1)

            # Print metrics
            print(f"Accuracy: {acc:.4f}")
            print(f"F1 Score (macro): {f1:.4f}")
            print("Classification Report:\n", classification_report(y_val, y_pred))

            # Confusion Matrix
            cm = confusion_matrix(y_val, y_pred)
            plt.figure(figsize=(6, 4))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
            plt.title(f"Confusion Matrix - {model_name} ({name})")
            plt.xlabel("Predicted")
            plt.ylabel("Actual")
            plt.tight_layout()
            plt.show()

        all_results[name] = perspective_results
        best_models_summary.append({
            "dataset": name,
            "best_model": best_model,
            "accuracy": best_acc,
            "f1_macro": best_f1
        })

    # Summary
    print("\n=== BEST MODELS SUMMARY (TF-IDF) ===")
    for entry in best_models_summary:
        print(f"- Dataset: {entry['dataset']}")
        print(f"  → Best Model: {entry['best_model']}")
        print(f"  → Accuracy: {entry['accuracy']:.4f}")
        print(f"  → F1 (Macro): {entry['f1_macro']:.4f}\n")

    return all_results


Essembling Voting

In [ ]:
def run_ensemble_tfidf_classification(datasets, max_features=10000):
    """
    Run ensemble classification using VotingClassifier with TF-IDF across different perspectives.
    """
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.model_selection import train_test_split, cross_val_score
    from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
    from sklearn.ensemble import VotingClassifier, RandomForestClassifier
    from sklearn.svm import LinearSVC
    from xgboost import XGBClassifier
    import seaborn as sns
    import matplotlib.pyplot as plt

    all_results = {}
    best_models_summary = []

    for name, df in datasets.items():
        print(f"\n=== Dataset: {name} ===")
        X = df["text_no_contractions"]
        y = df["label"]

        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

        # TF-IDF vectorization
        vectorizer = TfidfVectorizer(max_features=max_features)
        X_train_tfidf = vectorizer.fit_transform(X_train)
        X_val_tfidf = vectorizer.transform(X_val)

        # Convert to dense (XGBoost requirement)
        X_train_dense = X_train_tfidf.toarray()
        X_val_dense = X_val_tfidf.toarray()

        # Define base classifiers
        svm_clf = LinearSVC()
        rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
        xgb_clf = XGBClassifier(eval_metric='mlogloss', use_label_encoder=False, random_state=42)

        # Define voting classifier
        voting_clf = VotingClassifier(
            estimators=[
                ('SVM', svm_clf),
                ('RandomForest', rf_clf),
                ('XGBoost', xgb_clf)
            ],
            voting='hard'
        )

        # Train ensemble
        voting_clf.fit(X_train_dense, y_train)
        y_pred = voting_clf.predict(X_val_dense)

        # Metrics
        acc = accuracy_score(y_val, y_pred)
        f1 = f1_score(y_val, y_pred, average='macro')
        report = classification_report(y_val, y_pred)

        print(f"\nModel: Voting Ensemble ({name})")
        print(f"Accuracy: {acc:.4f}")
        print(f"F1 Score (macro): {f1:.4f}")
        print("Classification Report:\n", report)

        # Confusion matrix
        cm = confusion_matrix(y_val, y_pred)
        plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
        plt.title(f"Confusion Matrix - Voting Ensemble ({name})")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.tight_layout()
        plt.show()

        # Save best performance (across all datasets)
        globals()['Ensemble_TFIDF_accuracy'] = max(globals().get('Ensemble_TFIDF_accuracy', 0), acc)
        globals()['Ensemble_TFIDF_f1'] = max(globals().get('Ensemble_TFIDF_f1', 0), f1)

        # Save result
        all_results[name] = {
            "accuracy": acc,
            "f1_macro": f1,
            "report": report
        }

        best_models_summary.append({
            "dataset": name,
            "accuracy": acc,
            "f1_macro": f1
        })

    print("\n=== BEST RESULTS (Voting Ensemble - TFIDF) ===")
    for entry in best_models_summary:
        print(f"- Dataset: {entry['dataset']}")
        print(f"  → Accuracy: {entry['accuracy']:.4f}")
        print(f"  → F1 Score (Macro): {entry['f1_macro']:.4f}\n")

    return all_results


Word2vec

In [ ]:
def run_word2vec_classification(datasets, text_column="text_no_contractions", label_column="label"):
    """
    Runs classification using Word2Vec embeddings across different perspectives.
    """

    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import LinearSVC
    from sklearn.naive_bayes import GaussianNB
    from sklearn.ensemble import RandomForestClassifier
    from xgboost import XGBClassifier
    from gensim.models import Word2Vec
    from nltk.tokenize import word_tokenize
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
    import seaborn as sns
    import matplotlib.pyplot as plt
    import numpy as np

    models = {
        'LogisticRegression': LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000,
                                                 class_weight='balanced', random_state=42),
        'SVM': LinearSVC(class_weight='balanced', random_state=42),
        'NaiveBayes': GaussianNB(),
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
        'XGBoost': XGBClassifier(eval_metric='mlogloss', random_state=42)
    }

    all_results = {}
    best_overall_f1 = -1
    best_model_info = None

    for name, df in datasets.items():
        print(f"\n=== Dataset: {name} ===")

        X = df[text_column]
        y = df[label_column]

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

        # Tokenize and train Word2Vec
        tokenized_train = [word_tokenize(doc.lower()) for doc in X_train]
        w2v_model = Word2Vec(sentences=tokenized_train, vector_size=100, window=5, min_count=3, workers=4)

        def vectorize(doc):
            words = word_tokenize(doc.lower())
            vectors = [w2v_model.wv[word] for word in words if word in w2v_model.wv]
            return np.mean(vectors, axis=0) if vectors else np.zeros(w2v_model.vector_size)

        X_train_vec = np.array([vectorize(doc) for doc in X_train])
        X_test_vec = np.array([vectorize(doc) for doc in X_test])

        dataset_results = {}

        for model_name, model in models.items():
            print(f"\nModel: {model_name} (Word2Vec)")

            try:
                # Fit and predict
                model.fit(X_train_vec, y_train)
                y_pred = model.predict(X_test_vec)

                acc = accuracy_score(y_test, y_pred)
                f1 = f1_score(y_test, y_pred, average='macro')

                dataset_results[model_name] = {'accuracy': acc, 'f1_macro': f1}

                # Save results to globals for plotting
                globals()[f"{model_name}_Word2Vec_accuracy"] = max(
                    globals().get(f"{model_name}_Word2Vec_accuracy", 0), acc
                )
                globals()[f"{model_name}_Word2Vec_f1"] = max(
                    globals().get(f"{model_name}_Word2Vec_f1", 0), f1
                )

                print(f"Accuracy: {acc:.4f}")
                print(f"F1 Score (macro): {f1:.4f}")
                print("Classification Report:\n", classification_report(y_test, y_pred))

                # Save best model overall (across all datasets)
                if f1 > best_overall_f1:
                    best_overall_f1 = f1
                    best_model_info = {
                        'dataset': name,
                        'model': model_name,
                        'accuracy': acc,
                        'f1_macro': f1
                    }

                # Confusion matrix
                cm = confusion_matrix(y_test, y_pred)
                plt.figure(figsize=(6, 4))
                sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
                plt.title(f"Confusion Matrix - {model_name} ({name})")
                plt.xlabel("Predicted")
                plt.ylabel("Actual")
                plt.tight_layout()
                plt.show()

            except Exception as e:
                print(f"Model {model_name} failed on {name}: {e}")

        all_results[name] = dataset_results

    print("\n=== BEST MODEL OVERALL (Word2Vec) ===")
    if best_model_info:
        print(f"- Dataset: {best_model_info['dataset']}")
        print(f"  → Best Model: {best_model_info['model']}")
        print(f"  → Accuracy: {best_model_info['accuracy']:.4f}")
        print(f"  → F1 (Macro): {best_model_info['f1_macro']:.4f}")

    return all_results


Bow

In [ ]:
def run_bow_classification(datasets, text_column="text_no_contractions", label_column="label", vectorizer=None):
    """
    Runs classification using BoW across different perspectives.

    """

    from sklearn.feature_extraction.text import CountVectorizer

    models = {
        'LogisticRegression': LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000, random_state=42),
        'NaiveBayes': MultinomialNB(),
        'SVM': LinearSVC(random_state=42),
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
        'XGBoost': XGBClassifier(eval_metric='mlogloss', random_state=42)
    }

    # Initialize best scores
    best_scores = {
        model_name: {'f1': -1, 'accuracy': -1}
        for model_name in models.keys()
    }

    all_results = {}

    for dataset_name, df in datasets.items():
        print(f"\n=== Dataset: {dataset_name} ===")

        X = df[text_column]
        y = df[label_column]

        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

        if vectorizer is None:
            bow_vectorizer = CountVectorizer()
        else:
            bow_vectorizer = vectorizer

        X_train_bow = bow_vectorizer.fit_transform(X_train)
        X_val_bow = bow_vectorizer.transform(X_val)

        dataset_results = {}

        for name, model in models.items():
            print(f"\nModel: {name} (BoW)")

            # Fit & predict (XGBoost needs dense arrays)
            if name == 'XGBoost':
                model.fit(X_train_bow.toarray(), y_train)
                y_pred = model.predict(X_val_bow.toarray())
            else:
                model.fit(X_train_bow, y_train)
                y_pred = model.predict(X_val_bow)

            # Metrics
            accuracy = accuracy_score(y_val, y_pred)
            f1 = f1_score(y_val, y_pred, average='macro')

            # Save if best across all datasets
            if f1 > best_scores[name]['f1']:
                best_scores[name]['f1'] = f1
                best_scores[name]['accuracy'] = accuracy

                # Save with globals (final best only)
                globals()[f"{name}_BOW_f1"] = f1
                globals()[f"{name}_BOW_accuracy"] = accuracy

            # Print output
            print(f"Accuracy: {accuracy:.4f}")
            print(f"F1 Score (macro): {f1:.4f}")
            print("Classification Report:\n", classification_report(y_val, y_pred))

            # Confusion matrix
            plt.figure(figsize=(6, 4))
            cm = confusion_matrix(y_val, y_pred)
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
            plt.title(f"Confusion Matrix - {name} ({dataset_name})")
            plt.xlabel("Predicted")
            plt.ylabel("Actual")
            plt.tight_layout()
            plt.show()

            # Store results for this dataset
            dataset_results[name] = {'accuracy': accuracy, 'f1_macro': f1}

        all_results[dataset_name] = dataset_results

    print("\n=== Best Scores Across All Datasets (BoW) ===")
    for name, scores in best_scores.items():
        print(f"{name}: F1 = {scores['f1']:.4f} | Accuracy = {scores['accuracy']:.4f}")

    return all_results

SBert

In [ ]:
def run_sbert_classification(datasets, text_column="text_no_contractions", label_column="label"):
    """
    Runs classification using SBERT embeddings across the different perspectives.
    """

    from sentence_transformers import SentenceTransformer
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import LinearSVC
    from sklearn.naive_bayes import GaussianNB
    from sklearn.ensemble import RandomForestClassifier
    from xgboost import XGBClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
    import seaborn as sns
    import matplotlib.pyplot as plt
    import numpy as np

    sbert = SentenceTransformer('all-MiniLM-L6-v2')

    models = {
        'LogisticRegression': LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000,
                                                 class_weight='balanced', random_state=42),
        'SVM': LinearSVC(class_weight='balanced', random_state=42),
        'NaiveBayes': GaussianNB(),
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
        'XGBoost': XGBClassifier(eval_metric='mlogloss', use_label_encoder=False, random_state=42)
    }

    # Track best score per model across datasets
    best_scores = {model_name: {'f1': -1, 'accuracy': -1} for model_name in models}

    for dataset_name, df in datasets.items():
        print(f"\n=== Dataset: {dataset_name} ===")

        X = df[text_column].astype(str)
        y = df[label_column]

        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

        print("→ Encoding SBERT embeddings...")
        X_train_vec = sbert.encode(X_train.tolist(), show_progress_bar=True)
        X_val_vec = sbert.encode(X_val.tolist(), show_progress_bar=True)

        for name, model in models.items():
            print(f"\nModel: {name} (SBERT)")

            try:
                model.fit(X_train_vec, y_train)
                y_pred = model.predict(X_val_vec)

                accuracy = accuracy_score(y_val, y_pred)
                f1 = f1_score(y_val, y_pred, average='macro')

                # Save best score for this model across all datasets
                if f1 > best_scores[name]['f1']:
                    best_scores[name]['f1'] = f1
                    best_scores[name]['accuracy'] = accuracy

                    globals()[f"{name}_SBERT_f1"] = f1
                    globals()[f"{name}_SBERT_accuracy"] = accuracy

                print(f"Accuracy: {accuracy:.4f}")
                print(f"F1 Score (macro): {f1:.4f}")
                print("Classification Report:\n", classification_report(y_val, y_pred))

                # Confusion matrix
                cm = confusion_matrix(y_val, y_pred)
                plt.figure(figsize=(6, 4))
                sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
                plt.title(f"Confusion Matrix - {name} ({dataset_name} - SBERT)")
                plt.xlabel("Predicted")
                plt.ylabel("Actual")
                plt.tight_layout()
                plt.show()

            except Exception as e:
                print(f"Model {name} failed on {dataset_name}: {e}")

    print("\n=== BEST SBERT SCORES ACROSS ALL DATASETS ===")
    for name, score in best_scores.items():
        print(f"{name}: F1 = {score['f1']:.4f} | Accuracy = {score['accuracy']:.4f}")

Transformers

In [ ]:
def run_transformer_classifiers2(df, text_column="text_no_contractions", label_column="label"):
    """
    Runs classification using transformer models with custom 3-class label mapping:
    0 = negative, 1 = positive, 2 = neutral
    """

    # Split the data
    X = df[text_column].tolist()
    y = df[label_column]
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Define models
    transformer_models = {
        "FinBERT": "ProsusAI/finbert",
        "BERT": "nlptown/bert-base-multilingual-uncased-sentiment",
        "RoBERTa": "cardiffnlp/twitter-roberta-base-sentiment"
    }

    for model_name, model_path in transformer_models.items():
        print(f"\nModel: {model_name}")

        classifier = pipeline("text-classification", model=model_path, tokenizer=model_path, truncation=True)

        y_pred_raw = classifier(X_val, batch_size=16, truncation=True)

        # Convert predictions to custom labels: 0 = negative, 1 = positive, 2 = neutral
        if model_name == "FinBERT":
            label_map = {"positive": 1, "neutral": 2, "negative": 0}
            y_pred = [label_map.get(pred['label'].lower(), 2) for pred in y_pred_raw]

        elif model_name == "BERT":
            def map_bert_label(label):
                stars = int(label[0])
                if stars <= 2:
                    return 0  # negative
                elif stars == 3:
                    return 2  # neutral
                else:
                    return 1  # positive
            y_pred = [map_bert_label(pred['label']) for pred in y_pred_raw]

        elif model_name == "RoBERTa":
            # Assume LABEL_0 = negative, LABEL_1 = neutral, LABEL_2 = positive
            label_map = {"LABEL_0": 0, "LABEL_1": 2, "LABEL_2": 1}
            y_pred = [label_map.get(pred['label'], 2) for pred in y_pred_raw]

        # Metrics
        acc = accuracy_score(y_val, y_pred)
        f1 = f1_score(y_val, y_pred, average='macro')

        print(f"Accuracy: {acc:.4f}")
        print(f"F1 Score (macro): {f1:.4f}")
        print("Classification Report:\n", classification_report(y_val, y_pred))

        # Confusion Matrix
        cm = confusion_matrix(y_val, y_pred)
        plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
        plt.title(f"Confusion Matrix - {model_name}")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.tight_layout()
        plt.show()

In [ ]:
def run_transformer_classifiers(df, text_column="text_no_contractions", label_column="label"):
    from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
    from sklearn.model_selection import train_test_split
    import matplotlib.pyplot as plt
    import seaborn as sns
    from transformers import pipeline

    X = df[text_column].tolist()
    y = df[label_column]
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    transformer_models = {
        "FinBERT": "ProsusAI/finbert",
        "BERT": "nlptown/bert-base-multilingual-uncased-sentiment",
        "RoBERTa": "cardiffnlp/twitter-roberta-base-sentiment"
    }

    results = {}  # Collect scores here

    for model_name, model_path in transformer_models.items():
        print(f"\nModel: {model_name}")

        classifier = pipeline("text-classification", model=model_path, tokenizer=model_path, truncation=True)

        y_pred_raw = classifier(X_val, batch_size=16, truncation=True)

        if model_name == "FinBERT":
            label_map = {"positive": 1, "neutral": 2, "negative": 0}
            y_pred = [label_map.get(pred['label'].lower(), 2) for pred in y_pred_raw]

        elif model_name == "BERT":
            def map_bert_label(label):
                stars = int(label[0])
                if stars <= 2:
                    return 0
                elif stars == 3:
                    return 2
                else:
                    return 1
            y_pred = [map_bert_label(pred['label']) for pred in y_pred_raw]

        elif model_name == "RoBERTa":
            label_map = {"LABEL_0": 0, "LABEL_1": 2, "LABEL_2": 1}
            y_pred = [label_map.get(pred['label'], 2) for pred in y_pred_raw]

        acc = accuracy_score(y_val, y_pred)
        f1 = f1_score(y_val, y_pred, average='macro')
        results[model_name] = {'accuracy': acc, 'f1': f1}

        print(f"Accuracy: {acc:.4f}")
        print(f"F1 Score (macro): {f1:.4f}")
        print("Classification Report:\n", classification_report(y_val, y_pred))

        cm = confusion_matrix(y_val, y_pred)
        plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
        plt.title(f"Confusion Matrix - {model_name}")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.tight_layout()
        plt.show()

    return results


## **1.3. Loading and Reading the Data**



In [ ]:
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')
pd.set_option("display.max_columns", None)

In [ ]:
print(train_data.head())
print(train_data.tail())

In [ ]:
print(test_data.head())
print(test_data.tail())

# **2. Preprocessing**

In [ ]:
# More specified cleaning techniques

#Clean emojis from text
def strip_emoji(text):
    return re.sub(emoji.get_emoji_regexp(), r"", text) #remove emoji

#Remove punctuations, links, mentions and \r\n new line characters
def strip_all_entities(text):
    text = text.replace('\r', '').replace('\n', ' ').replace('\n', ' ').lower() #remove \n and \r and lowercase
    text = re.sub(r"(?:\@|https?\://)\S+", "", text) #remove links and mentions
    text = re.sub(r'[^\x00-\x7f]',r'', text) #remove non utf8/ascii characters such as '\x9a\x91\x97\x9a\x97'
    banned_list= string.punctuation + 'Ã'+'±'+'ã'+'¼'+'â'+'»'+'§'
    table = str.maketrans('', '', banned_list)
    text = text.translate(table)
    return text

#clean hashtags at the end of the sentence, and keep those in the middle of the sentence by removing just the # symbol
def clean_hashtags(tweet):
    new_tweet = " ".join(word.strip() for word in re.split('#(?!(?:hashtag)\b)[\w-]+(?=(?:\s+#[\w-]+)*\s*$)', tweet)) #remove last hashtags
    new_tweet2 = " ".join(word.strip() for word in re.split('#|_', new_tweet)) #remove hashtags symbol from words in the middle of the sentence
    return new_tweet2

#Filter special characters such as & and $ present in some words
def filter_chars(a):
    sent = []
    for word in a.split(' '):
        if ('$' in word) | ('&' in word):
            sent.append('')
        else:
            sent.append(word)
    return ' '.join(sent)

def remove_mult_spaces(text): # remove multiple spaces
    return re.sub("\s\s+" , " ", text)

In [ ]:
# Class inpired Cleaning

def clean(text_list, lemmatize, stemmer):

    updates = []

    for j in tqdm(text_list):

        text = j

        #LOWERCASE TEXT
        text = text.lower()

        #REMOVE NUMERICAL DATA and PUNCTUATION
        text = re.sub("[^a-zA-Z]"," ", text )
        # [^&] - Returns a match for any character EXCEPT &.
        # a-zA-Z - Returns a match for any character alphabetically between a and z, lower case or upper case.
        #text = re.sub("br", "", text)  # remove "br"
        text = re.sub(r'http\S+', '', text)  # remove URLs
        text = re.sub(f"[{re.escape(string.punctuation)}]", '', text)  # remove punctuation

        text = re.sub(r'\bco\b', '', text) #remove co in isolation
        text = re.sub(r'\s+', ' ', text).strip()


        #REMOVE STOPWORDS
        text = " ".join([word for word in text.split() if word not in stop])

        #Lemmatize
        if lemmatize:
            text = " ".join(lemma.lemmatize(word) for word in text.split())

        #Stemming
        if stemmer:
            text = " ".join(stemmer.stem(word) for word in text.split())

        updates.append(text)

    return updates

In [ ]:
# Merged Cleaning approaches

def clean_financial_tweet_with_tickers(text, lemmatize, stem):
    # 0. Extract cashtags (e.g., $TSLA)
    tickers = re.findall(r'\$\w+', text.upper())  # captures tickers in uppercase
    tickers = [t[1:] for t in tickers]  # remove the '$'

    # 1. Remove emojis
    text = re.sub(emoji.get_emoji_regexp(), "", text)

    # 2. Remove mentions, links, non-ascii chars, lowercase
    text = text.replace('\r', '').replace('\n', ' ').lower()
    text = re.sub(r"(?:\@|https?\://)\S+", "", text)
    text = re.sub(r'[^\x00-\x7f]',r'', text)

    # 3. Remove punctuation and weird chars
    banned_list = string.punctuation + 'Ã'+'±'+'ã'+'¼'+'â'+'»'+'§'
    text = text.translate(str.maketrans('', '', banned_list))

    # 4. Clean hashtags
    text = " ".join(word.strip() for word in re.split('#(?!(?:hashtag)\b)[\w-]+(?=(?:\s+#[\w-]+)*\s*$)', text))
    text = " ".join(word.strip() for word in re.split('#|_', text))

    # 5. Keep $ words intact (already extracted)
    # Optionally replace them with a placeholder like "TICKER" in text
    text = re.sub(r'\$\w+', '', text)

    # 6. Remove numbers and clean up
    text = re.sub("[^a-zA-Z]", " ", text)
    text = re.sub(r'\bco\b', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # 7. Stopwords
    text = " ".join(word for word in text.split() if word not in stop)

    # 8. Lemmatize
    if lemmatize:
        text = " ".join(lemma.lemmatize(word) for word in text.split())

    # 9. Stemming
    if stem:
        text = " ".join(stemmer.stem(word) for word in text.split())

    return text, tickers


## **2.1. Cleaning Approaches**

In [ ]:
# Imports and Preparation for the cleaning process
stop = set(stopwords.words('english'))
stemmer = SnowballStemmer('english')
lemma = WordNetLemmatizer()

### 2.1.1. Class inspired Cleaning

In [ ]:
updates = clean(train_data['text'], lemmatize = True, stemmer = False)
updates_t = clean(test_data['text'], lemmatize = True, stemmer = False)

In [ ]:
def update_df(dataframe, list_updated):
    dataframe.update(pd.DataFrame({"text": list_updated}))
    return dataframe

In [ ]:
# Df with class cleaning methods
cc_train_data = update_df(train_data, updates)
cc_test_data = update_df(test_data, updates)

In [ ]:
# Expanding the contractions on the text to improve tokenization accuracy
cc_train_data["text_no_contractions"] = [contractions.fix(doc) for doc in cc_train_data["text"]]
cc_test_data["text_no_contractions"] = [contractions.fix(doc) for doc in cc_test_data["text"]]


### 2.1.2. Tailored to Tweets and Social

In [ ]:
TS_train_data = train_data.copy()
TS_test_data = test_data.copy()

In [ ]:
TS_train_data

In [ ]:
texts_new = []
for t in TS_train_data.text:
    texts_new.append(remove_mult_spaces(filter_chars(clean_hashtags(strip_all_entities(t)))))

In [ ]:
texts_new_test = []
for t in TS_test_data.text:
    texts_new_test.append(remove_mult_spaces(filter_chars(clean_hashtags(strip_all_entities(t)))))

In [ ]:
TS_train_data['text_clean'] = texts_new
TS_test_data['text_clean'] = texts_new_test

### 2.1.3. Unified Methods

In [ ]:
updates_u = clean(train_data['text'], lemmatize = True, stemmer = False)
updates_u_t = clean(test_data['text'], lemmatize = True, stemmer = False)

In [ ]:
# Df with class cleaning methods
u_train_data = update_df(train_data, updates)
u_test_data = update_df(test_data, updates)

## **2.2. More Data Exploration**

In [ ]:
text_len = []
for text in TS_train_data.text_clean:
    tweet_len = len(text.split())
    text_len.append(tweet_len)

In [ ]:
TS_train_data['text_len'] = text_len

In [ ]:
text_len_test = []
for text in TS_test_data.text_clean:
    tweet_len = len(text.split())
    text_len_test.append(tweet_len)

In [ ]:
TS_test_data['text_len'] = text_len_test

Training tweets with less than 10 words

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot(x='text_len', data=TS_train_data[TS_train_data['text_len']<10], palette='Blues')
plt.title('Training tweets with less than 10 words')
plt.yticks([])
ax.bar_label(ax.containers[0])
plt.ylabel('count')
plt.xlabel('')
plt.show()

Test tweets with less than 10 words

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot(x='text_len', data=TS_test_data[TS_test_data['text_len']<10], palette='Blues')
plt.title('Test tweets with less than 10 words')
plt.yticks([])
ax.bar_label(ax.containers[0])
plt.ylabel('count')
plt.xlabel('')
plt.show()

## **2.3. More Preprocessing**

### 2.3.1 Word tokenize

In [ ]:
cc_train_data["token_text"] = [word_tokenize(doc) for doc in cc_train_data["text_no_contractions"]]
cc_test_data["token_text"] = [word_tokenize(doc) for doc in cc_test_data["text_no_contractions"]]


### 2.3.2. BertTokenizerFas

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

In [ ]:
token_lens = []

for txt in TS_train_data['text_clean'].values:
    tokens = tokenizer.encode(txt, max_length=512, truncation=True)
    token_lens.append(len(tokens))

max_len=np.max(token_lens)

In [ ]:
print(f"MAX TOKENIZED SENTENCE LENGTH: {max_len}")

In [ ]:
token_lens = []

for i,txt in enumerate(TS_train_data['text_clean'].values):
    tokens = tokenizer.encode(txt, max_length=512, truncation=True)
    token_lens.append(len(tokens))
    if len(tokens)>80:
        print(f"INDEX: {i}, TEXT: {txt}")

In [ ]:
TS_train_data['token_lens'] = token_lens


In [ ]:
TS_train_data = TS_train_data.sort_values(by='token_lens', ascending=False)
TS_train_data.head(5)

# **3. Feature Engineering**

## **3.1 Train and Evaluation Split**

In [ ]:
# Step 1: Store them in a dictionary
datasets = {
    "Class Inspired": cc_train_data,
    "Tailored for Tweets and Social": TS_train_data,
    "Unified": u_train_data
}

# Step 2: Define a consistent split function
from sklearn.model_selection import train_test_split

def split_data(df, text_column="text_no_contractions", label_column="label", test_size=0.2, seed=42):
    X = df[text_column]
    y = df[label_column]
    return train_test_split(X, y, test_size=test_size, random_state=seed, stratify=y)

# Step 3: Split and store them
splits = {}
for name, df in datasets.items():
    X_train, X_val, y_train, y_val = split_data(df)
    splits[name] = {
        "X_train": X_train,
        "X_val": X_val,
        "y_train": y_train,
        "y_val": y_val
    }


## **3.2 IF-IDF Vectorization**

In [ ]:
# Creating TF-IDF representations for the texts (limited to the top 5000 most relevant terms)
tfidf = TfidfVectorizer(max_features=10000)
# Fit on the training data
X_train_tfidf = tfidf.fit_transform(X_train)
# Fit on the test data
X_val_tfidf = tfidf.transform(X_val)

## **3.3 Bag of Words Vectorization**

In [ ]:
vectorizer = CountVectorizer()
X_train_bow = vectorizer.fit_transform(X_train)
X_val_bow = vectorizer.transform(X_val)

## **3.4 Word 2 Vectorization**

In [ ]:
all_w2v_models = {}
all_w2v_vectors = {}

for name, split in splits.items():
    X_train = split["X_train"]
    tokenized_texts = [word_tokenize(doc.lower()) for doc in X_train]

    w2v_model = Word2Vec(sentences=tokenized_texts, vector_size=100, window=5, min_count=3, workers=4)

    def vectorize(doc):
        vectors = [w2v_model.wv[word] for word in doc if word in w2v_model.wv]
        return np.mean(vectors, axis=0) if vectors else np.zeros(w2v_model.vector_size)

    # Tokenized training texts
    X_train_vectors = np.array([vectorize(doc) for doc in tokenized_texts])

    # Store per perspective
    all_w2v_models[name] = w2v_model
    all_w2v_vectors[name] = X_train_vectors


## **3.5. Sentence Transformer**

In [ ]:
from sentence_transformers import SentenceTransformer

# Load SBERT model (pretrained)
sbert = SentenceTransformer('all-MiniLM-L6-v2')

# Encode full dataset (or split first and encode separately)
X_train_vec = sbert.encode(X_train.tolist(), show_progress_bar=True)
X_val_vec = sbert.encode(X_val.tolist(), show_progress_bar=True)


# **4. Modeling**

## **4.1 Modeling with IF-IDF**

In [ ]:
results_tfidf = run_tfidf_classification(datasets, max_features=10000)

The best Model is the SVM, let´s try to do an assemble voting with SVM, Random Forest, and XGBoost and Compare it with SVM optimized via Grid Search

### 4.1.1 Essemble Voting

In [ ]:
run_ensemble_tfidf_classification(datasets, max_features=10000)

### 4.1.2 Grid Search SVM and Smoote

In [ ]:
# Defining the base classifiers to be used in the ensemble
svm_clf = LinearSVC()
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
xgb_clf = XGBClassifier(eval_metric='mlogloss', random_state=42)

# Creating a VotingClassifier with hard voting (majority rule)
voting_clf = VotingClassifier(
    estimators=[
        ('SVM', svm_clf),
        ('RandomForest', rf_clf),
        ('XGBoost', xgb_clf)
    ],
    voting='hard'
)

# Converting TF-IDF matrices to dense format (required for XGBoost)
X_train_dense = X_train_tfidf.toarray()
X_test_dense = X_val_tfidf.toarray()

# Training the ensemble
voting_clf.fit(X_train_dense, y_train)

# Making predictions on the test/validation set
y_pred_voting = voting_clf.predict(X_test_dense)

# Calculating accuracy and F1-score
Ensemble_TFIDF_accuracy = accuracy_score(y_val, y_pred_voting)
Ensemble_TFIDF_f1 = f1_score(y_val, y_pred_voting, average='macro')

# Output the results
print("\nModel: Voting Ensemble")
print("Accuracy:", Ensemble_TFIDF_accuracy)
print("F1 Score (macro):", Ensemble_TFIDF_f1)
print("Classification Report:\n", classification_report(y_val, y_pred_voting))

# Confusion matrix
cm_voting = confusion_matrix(y_val, y_pred_voting)
sns.heatmap(cm_voting, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - Voting Ensemble")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Cross-validation on the training set
cv_scores = cross_val_score(voting_clf, X_train_dense, y_train, cv=5, scoring='accuracy')
Ensemble_cv_accuracy = cv_scores.mean()

print("Cross-Validation Accuracy (mean):", Ensemble_cv_accuracy)


In [ ]:
# Define the parameter grid for SVM
param_grid = {
    'svm__C': [0.01, 0.1, 1, 10, 100, 1000],
    'svm__max_iter': [1000, 5000, 10000]
}

# Define the pipeline with SMOTE and LinearSVC
pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('svm', LinearSVC())
])

# Perform Grid Search with 5-fold cross-validation
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', verbose=1, n_jobs=-1)
grid_search.fit(X_train_tfidf, y_train)

# Get the best model from the grid search
best_model = grid_search.best_estimator_

# Evaluate on the training set
y_pred_train = best_model.predict(X_train_tfidf)
optimized_SVM_TFIDF_SMOTE_train_accuracy = accuracy_score(y_train, y_pred_train)

# Evaluate on the validation/test set
y_pred_test = best_model.predict(X_val_tfidf)
optimized_SVM_TFIDF_SMOTE_test_accuracy = accuracy_score(y_val, y_pred_test)
optimized_SVM_TFIDF_SMOTE_f1 = f1_score(y_val, y_pred_test, average='macro')

# Cross-validation accuracy from grid search
optimized_SVM_TFIDF_SMOTE_cv_accuracy = grid_search.best_score_

# Print results
print("\nBest hyperparameters found:", grid_search.best_params_)
print("Training Accuracy:", optimized_SVM_TFIDF_SMOTE_train_accuracy)
print("Cross-Validation Accuracy (mean):", optimized_SVM_TFIDF_SMOTE_cv_accuracy)
print("Test Accuracy:", optimized_SVM_TFIDF_SMOTE_test_accuracy)
print("F1 Score (macro):", optimized_SVM_TFIDF_SMOTE_f1)

# Classification report
print("\nClassification Report (Test Set):\n", classification_report(y_val, y_pred_test))

# Confusion matrix
cm = confusion_matrix(y_val, y_pred_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - Optimized SVM with TF-IDF + SMOTE")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


## **4.2 Modeling with Word 2 Vec**

In [ ]:
run_word2vec_classification(datasets, text_column="text_no_contractions", label_column="label")

## **4.3 Modeling with Bag of Words**

In [ ]:
run_bow_classification(datasets, text_column="text_no_contractions", label_column="label", vectorizer=None)

## **4.4. Modeling with SBert**

In [ ]:
run_sbert_classification(datasets=datasets, text_column="text_no_contractions", label_column="label")

## **4.5. Modeling with Transformers Classifiers**

In [ ]:
run_transformer_classifiers(TS_train_data)

# **5. Evaluation**

In [ ]:
# Define model names and vectorization methods
models = ['LogisticRegression', 'SVM', 'NaiveBayes', 'RandomForest', 'XGBoost', 'Ensemble', 'Optimized_SVM_SMOTE']
representations = ['BoW', 'TFIDF', 'Word2Vec', 'SBERT' ]

# Store accuracy values
accuracy_data = {
    'BoW': [
        globals().get('LogisticRegression_BOW_accuracy'),
        globals().get('SVM_BOW_accuracy'),
        globals().get('NaiveBayes_BOW_accuracy'),
        globals().get('RandomForest_BOW_accuracy'),
        globals().get('XGBoost_BOW_accuracy'),
        None,
        None
    ],
    'TFIDF': [
        globals().get('LogisticRegression_TFIDF_accuracy'),
        globals().get('SVM_TFIDF_accuracy'),
        globals().get('NaiveBayes_TFIDF_accuracy'),
        globals().get('RandomForest_TFIDF_accuracy'),
        globals().get('XGBoost_TFIDF_accuracy'),
        globals().get('Ensemble_TFIDF_accuracy'),
        globals().get('optimized_SVM_TFIDF_SMOTE_test_accuracy')
    ],
    'Word2Vec': [
        globals().get('LogisticRegression_Word2Vec_accuracy'),
        globals().get('SVM_Word2Vec_accuracy'),
        globals().get('NaiveBayes_Word2Vec_accuracy'),
        globals().get('RandomForest_Word2Vec_accuracy'),
        globals().get('XGBoost_Word2Vec_accuracy'),
        None,
        None
    ],

    'SBERT': [
        globals().get('LogisticRegression_SBERT_accuracy'),
        globals().get('SVM_SBERT_accuracy'),
        globals().get('NaiveBayes_SBERT_accuracy'),
        globals().get('RandomForest_SBERT_accuracy'),
        globals().get('XGBoost_SBERT_accuracy'),
        None,
        None
    ]}

# Store F1 scores - ONLY adding LogisticRegression columns
f1_data = {
    'BoW': [
        globals().get('LogisticRegression_BOW_f1'),
        globals().get('SVM_BOW_f1'),
        globals().get('NaiveBayes_BOW_f1'),
        globals().get('RandomForest_BOW_f1'),
        globals().get('XGBoost_BOW_f1'),
        None,
        None
    ],
    'TFIDF': [
        globals().get('LogisticRegression_TFIDF_f1'),
        globals().get('SVM_TFIDF_f1'),
        globals().get('NaiveBayes_TFIDF_f1'),
        globals().get('RandomForest_TFIDF_f1'),
        globals().get('XGBoost_TFIDF_f1'),
        globals().get('Ensemble_TFIDF_f1'),
        globals().get('optimized_SVM_TFIDF_SMOTE_f1')
    ],
    'Word2Vec': [
        globals().get('LogisticRegression_Word2Vec_f1'),
        globals().get('SVM_Word2Vec_f1'),
        globals().get('NaiveBayes_Word2Vec_f1'),
        globals().get('RandomForest_Word2Vec_f1'),
        globals().get('XGBoost_Word2Vec_f1'),
        None,
        None
    ],

    'SBERT': [
        globals().get('LogisticRegression_SBERT_f1'),
        globals().get('SVM_SBERT_f1'),
        globals().get('NaiveBayes_SBERT_f1'),
        globals().get('RandomForest_SBERTf1'),
        globals().get('XGBoost_SBERT_f1'),
        None,
        None

      ]
}

# Original colors and plotting function
colors = {
    'BoW': '#6699cc',
    'TFIDF': '#336699',
    'Word2Vec': '#003366',
     'SBERT': '#00264d'
}

def plot_model_comparison(data_dict, metric_label):
    x = np.arange(len(models))
    width = 0.2

    fig, ax = plt.subplots(figsize=(12, 7))

    for i, rep in enumerate(representations):
        values = data_dict[rep]
        offset = (i - 1) * width
        bars = ax.bar(x + offset, [v if v is not None else 0 for v in values],
                      width, label=rep, color=colors[rep])

        for bar in bars:
            height = bar.get_height()
            if height > 0:
                ax.annotate(f'{height:.2f}',
                            xy=(bar.get_x() + bar.get_width() / 2, height),
                            xytext=(0, 3), textcoords='offset points',
                            ha='center', va='bottom', fontsize=8)

    ax.set_ylabel(metric_label)
    ax.set_title(f'{metric_label} Comparison Across Models and Vectorizations')
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=45)
    ax.set_ylim(0, 1)
    ax.legend(title='Vectorization')

    plt.tight_layout()
    plt.show()

# Generate plots
plot_model_comparison(accuracy_data, 'Accuracy')
plot_model_comparison(f1_data, 'F1 Score (Macro)')

In [ ]:
# Transformers Models Results
transformer_results = {
    'FinBERT': {'accuracy': 0.6794, 'f1': 0.5496},
    'BERT': {'accuracy': 0.2546, 'f1': 0.2622},
    'RoBERTa': {'accuracy': 0.6412, 'f1': 0.4562}
}

models = list(transformer_results.keys())
accuracies = [transformer_results[model]['accuracy'] for model in models]
f1_scores = [transformer_results[model]['f1'] for model in models]

# Colors
accuracy_color = '#6699cc'
f1_color = '#003366'

x = np.arange(len(models))
width = 0.35

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x - width/2, accuracies, width, label='Accuracy',
               color=accuracy_color, linewidth=1, edgecolor='black')
bars2 = ax.bar(x + width/2, f1_scores, width, label='F1 Score',
               color=f1_color, linewidth=1, edgecolor='black')

for bars in [bars1, bars2]:
    for bar in bars:
        x0 = bar.get_x()
        bar_width = bar.get_width()
        ax.plot([x0, x0 + bar_width], [0, 0], color='black', linewidth=0.8)

# Value annotations
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=8)

# Dysplay the Plot
ax.set_ylabel('Score')
ax.set_title('Transformer Model Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 1)
ax.legend()

plt.tight_layout()
plt.show()
